# 04 · Model Baselines

Establish simple reference points before evaluating a more flexible ranking model.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

In [ ]:
metrics = json.loads((REPORTS / 'model_metrics.json').read_text())

## 1. Why accuracy is not a baseline

Fraud is close to one percent of the supplied base. Predicting every application as legitimate would appear accurate while recovering no fraud.

In [ ]:
audit = json.loads((REPORTS / 'data_audit.json').read_text())
audit['base']

## 2. Logistic regression

Class-weighted logistic regression provides an interpretable ranking baseline. It is not expected to capture complex interactions, but a more flexible model should demonstrate a clear benefit over it.

In [ ]:
pd.Series(metrics['models']['logistic_regression']['validation'])

In [ ]:
pd.Series(metrics['models']['logistic_regression']['test'])

## 3. Capacity view

In [ ]:
log_capacity = pd.DataFrame(metrics["models"]["logistic_regression"]["test_capacity"])
log_capacity[["review_share", "review_capacity", "fraud_captured", "precision_at_capacity", "recall_at_capacity"]]

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(log_capacity.review_share * 100, log_capacity.recall_at_capacity * 100, color=TEAL, marker="o", linewidth=2.5)
plt.xlabel("Applications reviewed (%)")
plt.ylabel("Fraud recovered (%)")
plt.title("Logistic baseline under fixed review capacity")
plt.show()

At three percent review capacity, the logistic model recovers 438 of 1,428 fraud cases in the final month. This becomes the minimum useful benchmark for model comparison.

## Conclusion

The logistic baseline ranks fraud meaningfully, but its performance leaves room for a model that captures nonlinear relationships without relying on audit-only fields.